# 🕵️ Sherlock Holmes RAG Chatbot

A conversational chatbot that can answer questions about **The Adventures of Sherlock Holmes** by Arthur Conan Doyle, using:

- **Groq** (free) → fast LLM inference with `llama-3.3-70b-versatile`
- **HuggingFace** (free, local) → sentence-transformer embeddings
- **LlamaIndex** → RAG orchestration (chunking, vector store, retrieval, chat engine)
- **Project Gutenberg** → public-domain source text (no copyright issues)

The bot keeps conversational memory, so you can ask follow-up questions naturally.

---
## 1. Installations 🛠️

In [ ]:
!pip install -qqq llama-index-core
!pip install -qqq llama-index-llms-groq
!pip install -qqq llama-index-embeddings-huggingface
!pip install -qqq sentence-transformers
!pip install -qqq python-dotenv

---
## 2. Load the Groq API key 🔑

**Never hardcode your API key in a notebook you plan to share.** Use one of the two options below.

In [ ]:
# --- Option A: Google Colab ---
# import os
# from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# --- Option B: Local (recommended) ---
# Create a `.env` file next to this notebook with one line:
#     GROQ_API_KEY=your_key_here
# and add `.env` to your .gitignore.
from dotenv import load_dotenv
load_dotenv()

import os
assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY not found — check your .env file"

---
## 3. Download the source text 📚

*The Adventures of Sherlock Holmes* from Project Gutenberg — 12 short stories, ~600KB of pure narrative gold.

In [ ]:
import os
import urllib.request

os.makedirs("data", exist_ok=True)

URL = "https://www.gutenberg.org/files/1661/1661-0.txt"
PATH = "data/sherlock_holmes.txt"

if not os.path.exists(PATH):
    urllib.request.urlretrieve(URL, PATH)
    print(f"Downloaded to {PATH}")
else:
    print(f"Already exists: {PATH}")

# Peek at the file size and a snippet
size_kb = os.path.getsize(PATH) / 1024
print(f"File size: {size_kb:.1f} KB")

with open(PATH, "r", encoding="utf-8") as f:
    text = f.read()
print(f"Total characters: {len(text):,}")
print("\n--- First 500 chars ---")
print(text[:500])

### 3.1 Trim Gutenberg boilerplate

Project Gutenberg wraps every book with a license header and footer. We strip these so the retriever only sees the actual story text.

In [ ]:
START_MARKER = "*** START OF THE PROJECT GUTENBERG EBOOK"
END_MARKER = "*** END OF THE PROJECT GUTENBERG EBOOK"

start_idx = text.find(START_MARKER)
end_idx = text.find(END_MARKER)

if start_idx != -1 and end_idx != -1:
    # Skip past the end of the START marker line
    start_idx = text.find("\n", start_idx) + 1
    clean_text = text[start_idx:end_idx].strip()
else:
    clean_text = text

# Save the cleaned version — LlamaIndex will load from this file
CLEAN_PATH = "data/sherlock_clean.txt"
with open(CLEAN_PATH, "w", encoding="utf-8") as f:
    f.write(clean_text)

print(f"Cleaned text: {len(clean_text):,} characters")
print("\n--- Opening lines ---")
print(clean_text[:400])

---
## 4. Set up the LLM 🧠

Groq gives us free, very fast inference on Llama 3.3 70B.

In [ ]:
from llama_index.llms.groq import Groq

llm = Groq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,   # low → factual, high → creative
    max_tokens=512,
)

# Quick sanity check
print(llm.complete("Say hello in one short sentence."))

---
## 5. Set up local embeddings 🔢

Embeddings turn text chunks into vectors so we can find the ones most relevant to a question. `all-MiniLM-L6-v2` runs locally, is tiny (~90MB), and works well for English prose.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    cache_folder="./embedding_model/",
)

# Sanity check — every vector should be 384-dim
vec = embed_model.get_text_embedding("A study in scarlet.")
print(f"Embedding dimension: {len(vec)}")

---
## 6. Build (or load) the vector index 🗂️

**How RAG works in three steps:**
1. **Chunk** the book into passages of ~512 tokens.
2. **Embed** each chunk into a vector.
3. **Store** the vectors so we can later retrieve the top-k most similar to any question.

Building takes ~1-2 minutes the first time. We persist it to disk so subsequent runs are instant.

In [ ]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
    Settings,
)

# Register the embedding model globally so all index operations use it
Settings.embed_model = embed_model
Settings.llm = llm
Settings.chunk_size = 512
Settings.chunk_overlap = 64  # small overlap so context is not chopped mid-sentence

INDEX_DIR = "./vector_index"

if not os.path.exists(INDEX_DIR):
    print("Building index from scratch...")
    documents = SimpleDirectoryReader(input_files=[CLEAN_PATH]).load_data()
    print(f"Loaded {len(documents)} document(s)")

    vector_index = VectorStoreIndex.from_documents(documents, show_progress=True)
    vector_index.storage_context.persist(persist_dir=INDEX_DIR)
    print(f"Index saved to {INDEX_DIR}")
else:
    print("Loading existing index from disk...")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_DIR)
    vector_index = load_index_from_storage(storage_context)
    print("Index loaded.")

---
## 7. Assemble the chat engine 💬

We combine four things:
- **Retriever** → finds the top-k chunks per question
- **Memory** → remembers earlier turns (up to 2000 tokens)
- **System prompts** → tell the LLM how to behave
- **ContextChatEngine** → the orchestrator

In [ ]:
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.base.llms.types import ChatMessage, MessageRole
from llama_index.core.memory import Memory

retriever = vector_index.as_retriever(similarity_top_k=3)

memory = Memory.from_defaults(token_limit=2000)

prefix_messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "You are a knowledgeable literary assistant specializing in "
            "'The Adventures of Sherlock Holmes' by Arthur Conan Doyle."
        ),
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content=(
            "Answer questions using ONLY the provided context from the book. "
            "If the answer is not in the context, say so honestly rather than guessing."
        ),
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Keep answers concise (2-4 sentences) unless asked for more detail.",
    ),
]

chatbot = ContextChatEngine(
    llm=llm,
    retriever=retriever,
    memory=memory,
    prefix_messages=prefix_messages,
)

---
## 8. Ask the detective 🔎

In [ ]:
response = chatbot.chat("Who is Irene Adler and why does Holmes remember her?")
print(response)

In [ ]:
response = chatbot.chat("What was the trick behind The Red-Headed League?")
print(response)

In [ ]:
# Follow-up question — the memory should carry context from above
response = chatbot.chat("Who was the villain behind it?")
print(response)

### 8.1 Inspect what the retriever pulled

Useful for debugging: which passages did the retriever consider most relevant?

In [ ]:
nodes = retriever.retrieve("What happens in The Speckled Band?")
for i, node in enumerate(nodes, 1):
    print(f"--- Chunk {i} (score: {node.score:.3f}) ---")
    print(node.text[:300], "...\n")

---
## 9. Interactive chat loop 🔁

Type your questions freely. Type `end` to quit, `reset` to clear memory.

In [ ]:
print("🕵️  Sherlock Holmes chatbot — ask me anything about the Adventures.")
print("Type 'end' to quit, 'reset' to clear memory.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() == "end":
        print("Goodbye!")
        break
    if user_input.lower() == "reset":
        chatbot.reset()
        print("Memory cleared.\n")
        continue

    response = chatbot.chat(user_input)
    print(f"\nHolmes-bot: {response}\n")

---
## 10. Ideas to extend this 🚀

- **Add more books** — drop *The Hound of the Baskervilles* or *A Study in Scarlet* into `data/`, delete `vector_index/`, and re-run cell 6.
- **Tune retrieval** — try `similarity_top_k=5`, or add a reranker.
- **Streamlit UI** — reuse the pattern from your `streamlit_rag_app.py`.
- **Cite sources** — show which chunk the answer came from in the UI.
- **Switch models** — try `llama-3.1-8b-instant` on Groq for faster (but less nuanced) answers.